In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## Install libraries

In [15]:
!pip install sentence-transformers faiss-cpu langchain-community


In [34]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_4665/3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [9]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "SwQhKFMxmDY"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

Hey everybody, welcome to the podcast. My guest today is Dr. Andrew Huberman. Andrew is a neuroscientist. He's a Neurobiology Professor at Stanford Medical School and McKnight Foundation and Pew Foundation Fellow and the founder of Huberman Lab, where he's involved in all kinds of really amazing breakthrough research on brain function, brain plasticity and brain regeneration. His work has been published in top journals like Nature. He's been featured in publications like Time, Scientific America and the BBC. And he's here today to discuss the brain, to discuss growth mindset, how to focus, how to navigate the stressful times and many other subjects. It's an incredible conversation. I think you guys are gonna enjoy it. I appreciate you watching make sure to hit that subscribe button and without further ado, this is me and Dr. Andrew Huberman. First of all, thanks for doing this. I appreciate you coming out. Yeah, my pleasure.
Long time coming. I'm glad we're doing it in person- Likewise

In [10]:
#fetched_transcript

In [11]:
transcript_list

[{'text': 'Hey everybody, welcome to the podcast.',
  'start': 0.4,
  'duration': 2.47},
 {'text': 'My guest today is Dr. Andrew Huberman.',
  'start': 2.87,
  'duration': 2.62},
 {'text': 'Andrew is a neuroscientist.', 'start': 5.49, 'duration': 2.5},
 {'text': "He's a Neurobiology Professor at Stanford Medical School",
  'start': 7.99,
  'duration': 3.88},
 {'text': 'and McKnight Foundation and Pew Foundation Fellow',
  'start': 11.87,
  'duration': 3.44},
 {'text': "and the founder of Huberman Lab, where he's involved",
  'start': 15.31,
  'duration': 2.37},
 {'text': 'in all kinds of really amazing breakthrough research',
  'start': 17.68,
  'duration': 2.26},
 {'text': 'on brain function, brain plasticity and brain regeneration.',
  'start': 19.94,
  'duration': 4.32},
 {'text': 'His work has been published in top journals like Nature.',
  'start': 24.26,
  'duration': 3.46},
 {'text': "He's been featured in publications",
  'start': 27.72,
  'duration': 1.34},
 {'text': 'like Tim

## Step 1b - Indexing (Text Splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [13]:
len(chunks)

184

In [ ]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
vector_store.index_to_docstore_id

{0: '60999e5a-97a6-4e8e-9265-2a159a1c7293',
 1: 'd9f581d2-7528-4236-a37e-9f4f3aaff9af',
 2: '73e77da3-e69a-4294-a093-58a739661996',
 3: '0661384f-1c4b-42cf-992c-68deec03e20b',
 4: '6c65afa6-7f4a-4ae6-afcf-dd884629e334',
 5: 'c6af8511-3df1-45af-b314-f4285dfb762f',
 6: 'f10b5561-ac97-4d94-ad37-029c1c5f55b2',
 7: '1afd3a1f-5456-46b9-a079-96b1fc71770e',
 8: '590b520e-6cbf-4362-b8d3-18794cb76683',
 9: '3e08b394-4a88-4d27-b7e6-a2c8a8eb4de6',
 10: '6ffaf200-1690-4acf-a9e7-4333165e6d5f',
 11: '2b4af59b-ec58-466f-87c3-3455a635c013',
 12: 'ea239438-4742-48df-8d28-c824f6d44359',
 13: '1ffc9888-034b-45b8-ac84-c3d943c00405',
 14: '54da9274-dc03-45ee-8593-2aa68ff1fdee',
 15: 'de858406-f2dc-4b4a-83f0-ce330bb9bac6',
 16: '474ba2df-b330-4e11-9780-27b187ae1fce',
 17: '856f246c-f768-44d0-a5f6-ab59ef04ec57',
 18: '9ba922dc-0aee-4695-a7b2-9531bdb21fcc',
 19: '9e167300-0a19-4ae5-8086-1797cfe3294a',
 20: 'bb62cfb3-906a-48df-a59f-7f3b99d3ac17',
 21: '51d4b375-8660-41ba-bd84-a6efc70aaf13',
 22: 'ae78663f-2017-

In [25]:
vector_store.get_by_ids(['700bc425-17dc-49d0-aa5c-4a2332f22423'])

[Document(id='700bc425-17dc-49d0-aa5c-4a2332f22423', metadata={}, page_content="consider it my obligation and I'm gonna keep going. Well, keep doing it, man. I appreciate the work that you're doing. I think it's really important work. We need it now more than ever. And it's cool that you're getting out there and sharing your wisdom with everybody. It's super empowering, so thanks man. Appreciate it.")]

## Step 2 - Retrieval

In [21]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [22]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b70272136e0>, search_kwargs={'k': 4})

In [26]:
retriever.invoke('How to change our brain')

[Document(id='1598258a-6242-4aef-8f6b-b0b591cc205b', metadata={}, page_content="Or three languages. Play a guitar or something like that. Yeah, without an accent, you know, three languages without an accent, it's remarkable and try and do that after age 25, it's very challenging. And so the brain is basically designed to be customized in the early part of life, and then to implement those algorithms in that circuitry for the rest of your, of its life. And so the brain can change in adulthood and it can change provided that there's an emphasis on some perceptual event. So in other words, if you wanna change your brain as an adult, let's say you wanna be less anxious, you wanna learn a new language, you wanna be more functional in some way, presumably, the key thing is to bring focus to some particular perception of something that's happening during the learning process. And the reason for that is that there's a neurochemical system involving acetylcholine. And it comes from these two li

## Step 3 - Augmentation

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [37]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [38]:
question          = "is the topic of human brain discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [39]:
retrieved_docs

[Document(id='9c861af2-358f-4f94-96d3-800cfa9ad5bc', metadata={}, page_content="in communication right now, culturally and socially, and it's fractured our society and it's not good, right? So what is going on neurologically with human beings that are attaching themselves and it's so self identifying with certain narratives that it's polarizing our population and preventing us from being able to just be together or united or agree upon what is true and what is not true and share a value system so that we can see our way through the challenges that we're facing right now, which many of which are an existential threat to the future of humanity and the planet. It's a huge problem, you articulated it beautifully. And I think neuroscience can offer a couple of insights into why it's happening and perhaps what we might do about it. So, one of the scientific results that I'm very intrigued by is in the 1960s, a guy named Robert Heath recorded from the human brain, there are people you can't d

In [40]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in communication right now, culturally and socially, and it's fractured our society and it's not good, right? So what is going on neurologically with human beings that are attaching themselves and it's so self identifying with certain narratives that it's polarizing our population and preventing us from being able to just be together or united or agree upon what is true and what is not true and share a value system so that we can see our way through the challenges that we're facing right now, which many of which are an existential threat to the future of humanity and the planet. It's a huge problem, you articulated it beautifully. And I think neuroscience can offer a couple of insights into why it's happening and perhaps what we might do about it. So, one of the scientific results that I'm very intrigued by is in the 1960s, a guy named Robert Heath recorded from the human brain, there are people you can't do this experiment nowadays, but skull popped off, my neurosurgery friends tell\

In [41]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [42]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in communication right now, culturally and socially, and it's fractured our society and it's not good, right? So what is going on neurologically with human beings that are attaching themselves and it's so self identifying with certain narratives that it's polarizing our population and preventing us from being able to just be together or united or agree upon what is true and what is not true and share a value system so that we can see our way through the challenges that we're facing right now, which many of which are an existential threat to the future of humanity and the planet. It's a huge problem, you articulated it beautifully. And I think neuroscience can offer a couple of insights into why it's happening and perhaps what we might do about it. So, one of the scientific results that I'm very int

## Step 4 - Generation

In [43]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of the human brain is discussed in the video.

Here's what was discussed:
*   **Neurological basis of societal fracturing:** The discussion begins by questioning what is happening neurologically that causes humans to attach to certain narratives, leading to societal polarization, inability to agree on truth, and lack of shared values, especially concerning existential threats. Neuroscience is suggested as a way to understand this.
*   **Robert Heath's 1960s experiments:** A scientific result from the early 1960s by Robert Heath is mentioned, where electrodes were placed deep into human brains. Participants could stimulate different brain areas and reported feeling things like being drunk, happy, or sexually aroused. They found one specific brain area that people repeatedly wanted to stimulate.
*   **Definition and function of the brain/nervous system:** The speaker defines the brain as part of the nervous system, including the brain, spinal cord, and connections with the

## Building a Chain

In [51]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [52]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [53]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [58]:
parallel_chain.invoke('how brain works')

{'context': "what's the starting point in the discussion around the brain? Yeah, so the brain and nervous system, which so it's like brain spinal cord connections with the body and back in. I don't distinguish between brain and mind. I think that's like an 80s discussion or earlier. And I think it would take us down the wrong track. So brain or mind to me is interchangeable. Mind, body, is kind of interchangeable because the brain is connected to the body and the body is connected to the brain, right? If I, you know, pinprick my hand and it hurts my brain registers it where it happens it's kind of an irrelevant discussion. I think we really need to just appreciate that the nervous system is designed to orchestrate all the processes in the body not just thinking and not just behavior and really can be divided into five things. So there's sensation and sensation is really bound or restricted by the receptors in the body. So receptors in the eye that perceive photons, light energy per se,

In [59]:
parser = StrOutputParser()

In [60]:
main_chain = parallel_chain | prompt | llm | parser

In [61]:
main_chain.invoke('Can you summarize the video')

'The provided text discusses several topics:\n\n1.  An anecdote about an individual named David who, despite disliking sharks, volunteered to be the first participant in an experiment involving technology and wiring people in. The speaker observed that David seems to convert adrenaline into understanding.\n2.  A reflection on the importance of children "foraging" for knowledge rather than passively receiving it.\n3.  The speaker\'s daily Instagram videos, which aim to teach neuroscience without complex language or acronyms, showing that scientists are normal people and explaining how the brain works and interfaces with psychology.\n4.  A description of a therapeutic technique involving lateralized eye movements. This method helps quiet the nervous system and allows people to process traumas, drawing a parallel to a deer\'s movement to alleviate stress and enhance situational awareness.'